# 内部状態の欠損要素を扱うには setdefault ではなく defaultdict を使う

自分で作成したのではない辞書を扱う場合、欠損キーの取り扱いに様々な方法があります。
in 式や KeyError 例外を使う方式よりも get メソッドを使う方がよいのですが、ユースケースによっては setdefault が最も短いコードになります。

例えば、世界中の国で自分が訪問した都市の記録を作るとします。国名に対応する都市名を含む集合を対応させる辞書を使うことにします。

In [1]:
visits = {
  'Mexico': {'Tulum', 'Puerto Vallarta'},
  'Japan': {'Hakone'},
}

In [2]:
visits.setdefault('France', set()).add('Arles') # 短い

if (japan := visits.get('Japan')) is not None: # 長い
    visits['Japan'] = japan = set()
japan.add('Kyoto')

print(visits)

{'Mexico': {'Tulum', 'Puerto Vallarta'}, 'Japan': {'Kyoto'}, 'France': {'Arles'}}


アクセスする辞書を与えられるのではなく、自分で作成できる場合ならどうでしょう？クラスの内部情報を辞書のインスタンスを使って管理する場合にはこちらの方が一般的です。次のコードでは、クラスでの上の例をヘルパーメソッドでラップして、辞書に格納されている動的な内部状態にアクセスします。

In [4]:
class Visits:
  def __init__(self):
    self.data = {}
  
  def add(self, country, city):
    city_set = self.data.setdefault(country, set())
    city_set.add(city)

この新たなクラスによって、プログラマは setdefault を正しく呼び出すための手間がなくなります。

In [6]:
visits = Visits()
visits.add('Russia', 'Yekaterinburg')
visits.add('Tanzania', 'Zanzibar')
print(visits.data)

{'Russia': {'Yekaterinburg'}, 'Tanzania': {'Zanzibar'}}


しかし、visits.add の実装にはまだ理想には達していません。setdefault の名前が混乱の元となっており、コードを新たに読む人には何が起こっているかがすぐにはつかめません。さらに、実装が効率的ではありません。国名がデータ辞書にあるかどうかにかかわらず、呼び出しのたびに新たな set インスタンスを作っているからです。

幸い、組み込み collections モジュールの defaultdict クラスには、キーが存在しない場合のデフォルト値を自動格納してこのユースケースを単純化する機能があります。キーがない場合に使うデフォルト値を返す関数を与えるだけでよいのです。defaultdict を使って Visits クラスを書き直します。

In [7]:
from collections import defaultdict

class Visits:
  def __init__(self):
    self.data = defaultdict(set)

  def add(self, country, city):
    self.data[country].add(city)

visits = Visits()
visits.add('England', 'Bath')
visits.add('England', 'London')
print(visits.data)

defaultdict(<class 'set'>, {'England': {'Bath', 'London'}})


add の実装が短く単純になりました。

## 覚えておくこと

- 辞書を作って任意のキー集合を処理するなら、問題に適しているなら組み込みモジュール collections の defaultdict インスタンスを使うべきだ。
- 辞書のキーを指定され、その辞書を自分で作成することができない状況なら、get を使って要素にアクセスしよう。しかし、より短いコードになる setdefault メソッドの使用を検討する状況もあるだろう。